In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS silver_external
URL 'abfss://silver@retailprojec1data.dfs.core.windows.net/'
WITH(
    STORAGE CREDENTIAL retail_project_cred
);

**- Setup:**

In [0]:
from pyspark.sql.functions import (
    col, current_date, current_timestamp, lit, when, 
    coalesce, hash, concat_ws, sha2
)
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

# Widget defaults
dbutils.widgets.text("storage_account", "retailprojec1data")
dbutils.widgets.text("bronze_container", "bronze")
dbutils.widgets.text("silver_container", "silver")

storage_account = dbutils.widgets.get("storage_account")
bronze_container = dbutils.widgets.get("bronze_container")
silver_container = dbutils.widgets.get("silver_container")

# Paths
bronze_customers_path = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/customers_delta/"
silver_dim_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/dim_customers/"

print(f"Bronze:  {bronze_customers_path}")
print(f"Silver:  {silver_dim_path}")

**- Reading from the bronze**

In [0]:
bronze_customers = spark.read.format("delta").load(bronze_customers_path)
print(f"Bronze customers: {bronze_customers.count()}")
bronze_customers.show(3, truncate=False)

**- Defining tracked columns**

In [0]:
from pyspark.sql.functions import (
    col, current_date, current_timestamp, lit, when, 
    coalesce, hash, concat_ws, sha2, to_date
)

# Columns that trigger SCD2 versioning when changed
TRACKED_COLUMNS = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "city",
    "state",
    "loyalty_tier"
]

# Prepare incoming with row_hash for change detection
incoming = (bronze_customers
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "city",
        "state",
        "loyalty_tier",
        # Parse signup_date from string to date, trying multiple formats
        coalesce(
            to_date(col("signup_date"), "M/d/yyyy"),
            to_date(col("signup_date"), "MM/dd/yyyy"),
            to_date(col("signup_date"), "yyyy-MM-dd"),
            to_date(col("signup_date"))  # Default ISO format
        ).alias("signup_date"),
    )
    .withColumn(
        "row_hash",
        hash(concat_ws("||", *[coalesce(col(c), lit("NULL")).cast("string") for c in TRACKED_COLUMNS]))
    )
)

print(f"Incoming rows: {incoming.count()}")
incoming.printSchema()
incoming.show(3, truncate=False)

**- Quarantine bad_dates**

In [0]:
# Check for parse failures
parse_failures = incoming.filter(col("signup_date").isNull())
failure_count = parse_failures.count()

if failure_count > 0:
    print(f"⚠️ {failure_count} rows failed date parsing")
    
    # Write to quarantine
    quarantine_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/_quarantine/dim_customers_bad_dates/"
    
    (parse_failures
        .withColumn("quarantine_reason", lit("signup_date_parse_failed"))
        .withColumn("quarantine_ts", current_timestamp())
        .write.format("delta").mode("append").save(quarantine_path))
    
    print(f"   Wrote {failure_count} rows to {quarantine_path}")
    
    # Filter them OUT of incoming so they don't pollute Silver
    incoming = incoming.filter(col("signup_date").isNotNull())
    print(f"   Continuing with {incoming.count()} clean rows")
else:
    print("✅ All rows parsed dates successfully")

**- checking if table exists or not**

In [0]:
silver_exists = False

try:
    test_df = spark.read.format("delta").load(silver_dim_path)
    row_count = test_df.count()
    
    if row_count > 0:
        silver_exists = True
        print(f"✅ Silver dim_customers EXISTS with {row_count} rows")
    else:
        silver_exists = False
        print(f"⚠️ Silver location exists but is EMPTY (0 rows) — treating as new")
        
except Exception as e:
    silver_exists = False
    print(f"⚠️ Cannot read Silver dim_customers: {type(e).__name__}")
    print(f"   {str(e)[:200]}")

print(f"\nsilver_exists = {silver_exists}")
print(f"Will {'apply MERGE' if silver_exists else 'INITIALIZE Silver'}")

if not silver_exists:
    # Generate hash-based customer_sk from customer_id + effective_from
    # Same inputs always produce the same hash
    
    initial = (incoming
        .withColumn("effective_from", col("signup_date"))
        .withColumn(
    "customer_sk",
    sha2(
        concat_ws("||", 
            col("customer_id").cast("string"),
            col("effective_from").cast("string"),
            col("row_hash").cast("string")
        ),
        256
    )
)
        .withColumn("effective_to", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("silver_ingestion_ts", current_timestamp())
    )
    
    # Reorder columns
    final_columns = (
        ["customer_sk", "customer_id"] 
        + TRACKED_COLUMNS 
        + ["signup_date", "effective_from", "effective_to", "is_current", "row_hash", "silver_ingestion_ts"]
    )
    initial = initial.select(*final_columns)
    
    initial.write.format("delta").mode("overwrite").save(silver_dim_path)
    
    initial_count = spark.read.format("delta").load(silver_dim_path).count()
    print(f"✅ Silver dim_customers initialized with {initial_count} rows")
    print(f"\nSample customer_sk values:")
    initial.select("customer_id", "customer_sk").show(3, truncate=False)

**- if table exists doing incremental load**

In [0]:
if silver_exists:
    silver_full = spark.read.format("delta").load(silver_dim_path)
    silver_current = silver_full.filter(col("is_current") == True)
    
    # 1. Detect changed customers (row_hash differs from current Silver)
    changes = (incoming.alias("new")
        .join(silver_current.alias("old"), "customer_id")
        .filter(col("new.row_hash") != col("old.row_hash"))
        .select(col("new.customer_id"))
    )
    
    changed_ids = [row.customer_id for row in changes.collect()]
    print(f"Customers with changes: {len(changed_ids)}")
    
    # 2. Detect new customers (in incoming but not in Silver)
    new_customers_df = (incoming.alias("new")
        .join(silver_current.alias("old"), "customer_id", "left_anti")
    )
    
    new_count = new_customers_df.count()
    print(f"Brand new customers: {new_count}")
    
    # 3. Close out old rows for changed customers
    if len(changed_ids) > 0:
        silver_table = DeltaTable.forPath(spark, silver_dim_path)
        silver_table.update(
            condition = (col("customer_id").isin(changed_ids)) & (col("is_current") == True),
            set = {
                "effective_to": "current_date()",
                "is_current": "false"
            }
        )
        print(f"✅ Closed out {len(changed_ids)} old SCD2 rows")
    
    # 4. Insert new SCD2 rows
    all_new_ids = changed_ids + [row.customer_id for row in new_customers_df.select("customer_id").collect()]
    
    if len(all_new_ids) > 0:
        rows_to_insert = incoming.filter(col("customer_id").isin(all_new_ids))
        
        # Generate hash-based customer_sk
        rows_with_sk = (rows_to_insert
            .withColumn("effective_from", current_date())
            .withColumn(
    "customer_sk",
    sha2(
        concat_ws("||", 
            col("customer_id").cast("string"),
            col("effective_from").cast("string"),
            col("row_hash").cast("string")
        ),
        256
    )
)
            .withColumn("effective_to", lit(None).cast("date"))
            .withColumn("is_current", lit(True))
            .withColumn("silver_ingestion_ts", current_timestamp())
        )
        
        # Reorder columns to match existing schema
        final_columns = (
            ["customer_sk", "customer_id"] 
            + TRACKED_COLUMNS 
            + ["signup_date", "effective_from", "effective_to", "is_current", "row_hash", "silver_ingestion_ts"]
        )
        rows_with_sk = rows_with_sk.select(*final_columns)
        
        rows_with_sk.write.format("delta").mode("append").save(silver_dim_path)
        print(f"✅ Inserted {rows_with_sk.count()} new SCD2 rows with hash-based customer_sk")
    else:
        print("No new rows to insert")

**- Verifying the data**

In [0]:
silver_dim = spark.read.format("delta").load(silver_dim_path)

print("=" * 60)
print("dim_customers SCD2 Summary (Hash-Based Surrogate Key)")
print("=" * 60)
print(f"Total rows:          {silver_dim.count()}")
print(f"Current rows:        {silver_dim.filter(col('is_current') == True).count()}")
print(f"Historical rows:     {silver_dim.filter(col('is_current') == False).count()}")
print(f"Unique customers:    {silver_dim.select('customer_id').distinct().count()}")
print(f"Unique customer_sk:  {silver_dim.select('customer_sk').distinct().count()}")

print("\nSample (showing the hash-based surrogate keys):")
(silver_dim
    .orderBy("customer_id", "effective_from")
    .select("customer_sk", "customer_id", "loyalty_tier", "effective_from", "effective_to", "is_current")
    .show(10, truncate=False))

# Verify hashes are unique (this is the production safety check)
total = silver_dim.count()
unique_sk = silver_dim.select("customer_sk").distinct().count()
if total == unique_sk:
    print(f"✅ All {total} customer_sk values are unique (no hash collisions)")
else:
    print(f"⚠️ WARNING: Hash collision detected! {total} rows but only {unique_sk} unique customer_sk")

**- Exit**

In [0]:
total = silver_dim.count()
current = silver_dim.filter(col("is_current") == True).count()
historical = silver_dim.filter(col("is_current") == False).count()

dbutils.notebook.exit(f"total_{total}_current_{current}_historical_{historical}")